# SmolVLM 256M experiment record
Initial single-image language-only LoRA; smoke test and full training are separate cells.

Original: `EdgeVLM_finetuning_fixed.ipynb`. Numeric/text outputs are retained where present. Embedded visual outputs and image-path previews are removed for publication. Execution counts are not fabricated. See README.md and the source manifest.

This is a historical server notebook, not a portable CLI. Review paths, dependencies, adapter initialization and optional export cells before executing. No dataset or weights are included.


# EdgeVLM-Drive SmolVLM fine-tuning

Run the cells from top to bottom after restarting the kernel. The notebook keeps one CSV-loading cell and one model-loading cell.


In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

print("TensorFlow disabled for Transformers.")

TensorFlow disabled for Transformers.


In [2]:
from pathlib import Path

print("Notebook directory:", Path.cwd())

candidates = [
    Path.cwd() / "splits",
    Path.cwd().parent / "splits",
    Path("/home/oai/share/splits"),
    Path("/home/jovyan/splits"),
    Path("/mnt/data/splits"),
    Path("/workspace/splits"),
]

for folder in candidates:
    print(folder, "exists:", folder.exists())
    if folder.exists():
        print("Files:", [p.name for p in folder.iterdir()])

Notebook directory: /home/jovyan/EdgeVLM_finetune
/home/jovyan/EdgeVLM_finetune/splits exists: False
/home/jovyan/splits exists: True
Files: ['val.csv', '.ipynb_checkpoints', 'train.csv', 'test.csv']
/home/oai/share/splits exists: False
/home/jovyan/splits exists: True
Files: ['val.csv', '.ipynb_checkpoints', 'train.csv', 'test.csv']
/mnt/data/splits exists: False
/workspace/splits exists: False


Read the CSV files

In [3]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/home/jovyan/EdgeVLM_finetune")
SPLITS_DIR = Path("/home/jovyan/splits")

TRAIN_CSV = SPLITS_DIR / "train.csv"
VAL_CSV = SPLITS_DIR / "val.csv"
TEST_CSV = SPLITS_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Columns:", train_df.columns.tolist())
print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nFirst frame path:")
print(train_df.iloc[0]["frame_path"])

Columns: ['frame_path', 'label', 'source_dataset', 'group', 'subject']
Training rows: 16202
Validation rows: 3941
Test rows: 3579

Training labels:
label
NORMAL          9390
PHONE_USAGE     4565
LOOKING_AWAY    1514
DROWSY           733
Name: count, dtype: int64

First frame path:


In [4]:
sample_paths = train_df["frame_path"].dropna().astype(str).head(100)

candidate_roots = [
    Path("/home/jovyan"),
    Path("/home/jovyan/dmd_data_downloads"),
    Path("/home/jovyan/dmd_distraction_downloads"),
    Path("/home/jovyan/scratchdata"),
    Path("/home/jovyan/build-dataset-from-video"),
]

for root in candidate_roots:
    if root.exists():
        found = sum((root / Path(item)).is_file() for item in sample_paths)
        print(root, "->", found, "of", len(sample_paths), "images found")

/home/jovyan -> 100 of 100 images found
/home/jovyan/dmd_data_downloads -> 0 of 100 images found
/home/jovyan/dmd_distraction_downloads -> 0 of 100 images found
/home/jovyan/scratchdata -> 0 of 100 images found
/home/jovyan/build-dataset-from-video -> 0 of 100 images found


Check data quality and split leakage

In [5]:
FRAME_ROOT = Path("/home/jovyan")

In [6]:
from pathlib import Path

datasets = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

# Check missing images
for name, df in datasets.items():
    missing = []

    for frame_path in df["frame_path"].astype(str):
        full_path = FRAME_ROOT / Path(frame_path)
        if not full_path.is_file():
            missing.append(str(full_path))

    print(name, "missing images:", len(missing))
    if missing:
        print("Example missing file:", missing[0])

# Check whether groups or subjects overlap between splits
for column in ["group", "subject"]:
    split_values = {
        name: set(df[column].astype(str).unique())
        for name, df in datasets.items()
    }

    print("\nChecking:", column)

    for first, second in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]:
        overlap = split_values[first].intersection(split_values[second])
        print(first, "and", second, "overlap:", len(overlap))

train missing images: 0
validation missing images: 0
test missing images: 0

Checking: group
train and validation overlap: 3
train and test overlap: 2
validation and test overlap: 1

Checking: subject
train and validation overlap: 0
train and test overlap: 0
validation and test overlap: 0


In [7]:
for name, df in datasets.items():
    print("\n", name)
    print("Groups:", sorted(df["group"].astype(str).unique()))
    print(
        df.groupby("group")["subject"]
        .nunique()
        .rename("number_of_subjects")
    )


 train
Groups: ['gA', 'gB', 'gC', 'gE', 'gF', 'gZ']
group
gA    1
gB    1
gC    1
gE    1
gF    1
gZ    3
Name: number_of_subjects, dtype: int64

 validation
Groups: ['gA', 'gB', 'gE']
group
gA    1
gB    1
gE    1
Name: number_of_subjects, dtype: int64

 test
Groups: ['gB', 'gC']
group
gB    2
gC    1
Name: number_of_subjects, dtype: int64


Check the notebook environment

In [8]:
import sys
import importlib.util

print("Python:", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory (GB):",
              round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except ImportError:
    print("PyTorch is not installed")

for package in ["transformers", "tokenizers", "peft", "accelerate", "datasets"]:
    installed = importlib.util.find_spec(package) is not None
    print(package, "installed:", installed)

Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]
PyTorch: 2.11.0+cu126
CUDA available: True
GPU: NVIDIA A16
GPU memory (GB): 14.61
transformers installed: True
tokenizers installed: True
peft installed: True
accelerate installed: True
datasets installed: True


Install fine-tuning dependencies

In [9]:
%pip install -q \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "peft>=0.13" \
    "accelerate>=1.0" \
    "datasets>=3.0" \
    "pandas>=2.0" \
    "Pillow>=10.0" \
    "safetensors>=0.4" \
    "sentencepiece" \
    "tqdm"


[notice] A new release of pip is available: 25.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import torch
import transformers
import tokenizers
import peft
import accelerate
import datasets

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("PEFT:", peft.__version__)

PyTorch: 2.11.0+cu126
CUDA: True
Transformers: 4.46.3
Tokenizers: 0.20.3
PEFT: 0.20.0


Download and load SmolVLM

In [11]:
from pathlib import Path
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq

PROJECT_ROOT = Path("/home/jovyan/EdgeVLM_finetune")
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
MODEL_CACHE = PROJECT_ROOT / "model_cache"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

print("Loading processor...")
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    cache_dir=str(MODEL_CACHE)
)

print("Loading model...")
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    cache_dir=str(MODEL_CACHE)
)

model = model.to("cuda")
model.eval()

print("Model loaded successfully")
print("Device:", next(model.parameters()).device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Loading processor...


Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


Loading model...
Model loaded successfully
Device: cuda:0
Parameters: 256484928


Test one image

In [12]:
from PIL import Image
from pathlib import Path
import torch

FRAME_ROOT = Path("/home/jovyan")

sample = train_df.iloc[0]
image_path = FRAME_ROOT / sample["frame_path"]
image = Image.open(image_path).convert("RGB")

prompt = """Classify the driver's visible state as exactly one label:
NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE.

Return only one label."""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt},
        ],
    }
]

chat_prompt = processor.apply_chat_template(
    messages,
    add_generation_prompt=True
)

inputs = processor(
    text=chat_prompt,
    images=[image],
    return_tensors="pt",
    size={"longest_edge": 512},
    max_image_size={"longest_edge": 512},
)

inputs = inputs.to("cuda")

with torch.no_grad():
    generated = model.generate(
        **inputs,
        max_new_tokens=12,
        do_sample=False
    )

input_length = inputs["input_ids"].shape[-1]
generated_only = generated[:, input_length:]

prediction_text = processor.batch_decode(
    generated_only,
    skip_special_tokens=True
)[0].strip()

print("Image:", image_path)
print("Ground-truth label (evaluation only):", sample["label"])
print("Model output:", prediction_text)

Ground-truth label (evaluation only): NORMAL
Model output: PHONE_USAGE.


create the training dataset wrapper

In [13]:
from torch.utils.data import Dataset
from PIL import Image

INSTRUCTION = (
    "Classify the driver's visible state. "
    "Return exactly one label from: NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE. "
    "Do not provide an explanation or punctuation."
)

class DriverStateDataset(Dataset):
    def __init__(self, dataframe, frame_root):
        self.dataframe = dataframe.reset_index(drop=True)
        self.frame_root = frame_root

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.frame_root / row["frame_path"]

        with Image.open(image_path) as image_file:
            image = image_file.convert("RGB").copy()

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": INSTRUCTION},
                ],
            }
        ]

        target_label = str(row["label"]).strip()

        return {
            "image": image,
            "messages": messages,
            "target": target_label,
            "path": str(image_path),
        }

train_dataset = DriverStateDataset(train_df, FRAME_ROOT)
val_dataset = DriverStateDataset(val_df, FRAME_ROOT)

example = train_dataset[0]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Example target:", example["target"])
print("Example image:", example["path"])
print("Target is not included in the model input yet.")

Training examples: 16202
Validation examples: 3941
Example target: NORMAL
Target is not included in the model input yet.


Create the training collator

In [14]:
import torch
from torch.utils.data import DataLoader

class SmolVLMDataCollator:
    def __init__(self, processor, image_size=512):
        self.processor = processor
        self.image_size = image_size

    def __call__(self, features):
        images = [item["image"] for item in features]
        full_texts = []
        prompt_texts = []

        for item in features:
            user_messages = item["messages"]

            full_messages = user_messages + [
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": item["target"]}
                    ],
                }
            ]

            prompt_texts.append(
                self.processor.apply_chat_template(
                    user_messages,
                    add_generation_prompt=True,
                    tokenize=False,
                )
            )

            full_texts.append(
                self.processor.apply_chat_template(
                    full_messages,
                    add_generation_prompt=False,
                    tokenize=False,
                )
            )

        # One image belongs to each text example
        images_for_processor = [[image] for image in images]

        processor_arguments = {
            "text": full_texts,
            "images": images_for_processor,
            "return_tensors": "pt",
            "padding": True,
            "truncation": True,
            "size": {"longest_edge": self.image_size},
            "max_image_size": {"longest_edge": self.image_size},
        }

        batch = self.processor(**processor_arguments)

        # Process prompts separately to find where the answer begins
        prompt_batch = self.processor(
            text=prompt_texts,
            images=images_for_processor,
            return_tensors="pt",
            padding=True,
            truncation=True,
            size={"longest_edge": self.image_size},
            max_image_size={"longest_edge": self.image_size},
        )

        labels = batch["input_ids"].clone()

        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)

        for row_number, prompt_length in enumerate(prompt_lengths):
            labels[row_number, :int(prompt_length)] = -100

        # Ignore padding tokens when calculating loss
        pad_token_id = self.processor.tokenizer.pad_token_id

        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        batch["labels"] = labels

        return batch


collator = SmolVLMDataCollator(processor)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collator,
)

test_batch = next(iter(train_loader))

print("Batch created successfully")
for key, value in test_batch.items():
    if hasattr(value, "shape"):
        print(key, value.shape)
    else:
        print(key, type(value))

print("Trainable target tokens:", (test_batch["labels"] != -100).sum().item())

Batch created successfully
pixel_values torch.Size([2, 1, 3, 512, 512])
pixel_attention_mask torch.Size([2, 1, 512, 512])
input_ids torch.Size([2, 128])
attention_mask torch.Size([2, 128])
labels torch.Size([2, 128])
Trainable target tokens: 12


Inspect LoRA target layers

In [15]:
import torch

attention_layers = []

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        if name.endswith(("q_proj", "k_proj", "v_proj", "o_proj")):
            attention_layers.append(name)

print("Number of attention layers:", len(attention_layers))

print("\nFirst attention layers:")
for name in attention_layers[:30]:
    print(name)

print("\nPossible text-model prefixes:")
prefixes = set()

for name in attention_layers:
    if ".layers." in name:
        prefixes.add(name.split(".layers.")[0])

for prefix in sorted(prefixes):
    print(prefix)
    

Number of attention layers: 156

First attention layers:
model.vision_model.encoder.layers.0.self_attn.k_proj
model.vision_model.encoder.layers.0.self_attn.v_proj
model.vision_model.encoder.layers.0.self_attn.q_proj
model.vision_model.encoder.layers.1.self_attn.k_proj
model.vision_model.encoder.layers.1.self_attn.v_proj
model.vision_model.encoder.layers.1.self_attn.q_proj
model.vision_model.encoder.layers.2.self_attn.k_proj
model.vision_model.encoder.layers.2.self_attn.v_proj
model.vision_model.encoder.layers.2.self_attn.q_proj
model.vision_model.encoder.layers.3.self_attn.k_proj
model.vision_model.encoder.layers.3.self_attn.v_proj
model.vision_model.encoder.layers.3.self_attn.q_proj
model.vision_model.encoder.layers.4.self_attn.k_proj
model.vision_model.encoder.layers.4.self_attn.v_proj
model.vision_model.encoder.layers.4.self_attn.q_proj
model.vision_model.encoder.layers.5.self_attn.k_proj
model.vision_model.encoder.layers.5.self_attn.v_proj
model.vision_model.encoder.layers.5.self_a

Add LoRA

In [16]:
import torch
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

# Find attention layers only inside the text model
text_attention_targets = []

for name, module in model.named_modules():
    if (
        name.startswith("model.text_model.")
        and isinstance(module, torch.nn.Linear)
        and name.endswith(("q_proj", "k_proj", "v_proj", "o_proj"))
    ):
        text_attention_targets.append(name)

text_attention_targets = sorted(text_attention_targets)

print("Text-model LoRA targets:", len(text_attention_targets))
print("First targets:")
for name in text_attention_targets[:10]:
    print(name)

if not isinstance(model, PeftModel):
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=text_attention_targets,
    )

    model.config.use_cache = False
    model = get_peft_model(model, lora_config)

model.train()
model.print_trainable_parameters()

Text-model LoRA targets: 120
First targets:
model.text_model.layers.0.self_attn.k_proj
model.text_model.layers.0.self_attn.o_proj
model.text_model.layers.0.self_attn.q_proj
model.text_model.layers.0.self_attn.v_proj
model.text_model.layers.1.self_attn.k_proj
model.text_model.layers.1.self_attn.o_proj
model.text_model.layers.1.self_attn.q_proj
model.text_model.layers.1.self_attn.v_proj
model.text_model.layers.10.self_attn.k_proj
model.text_model.layers.10.self_attn.o_proj
trainable params: 921,600 || all params: 257,406,528 || trainable%: 0.3580


Training smoke test

In [17]:
# This is a wiring test only (three optimizer steps).
from pathlib import Path
from transformers import TrainingArguments, Trainer

# Define this again here so the cell gives a clear error instead of a NameError
# if somebody runs it directly after a kernel restart.
PROJECT_ROOT = Path("/home/jovyan/EdgeVLM_finetune")
SMOKE_OUTPUT = PROJECT_ROOT / "lora_smoke_test"
SMOKE_OUTPUT.mkdir(parents=True, exist_ok=True)

missing = [name for name in ["model", "train_dataset", "collator"] if name not in globals()]
if missing:
    raise RuntimeError(f"Run the previous cells first; missing: {missing}")

training_args = TrainingArguments(
    output_dir=str(SMOKE_OUTPUT),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=3,
    learning_rate=1e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=1,
    save_strategy="no",
    evaluation_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collator,
)

print("Starting smoke test...")
train_result = trainer.train()
print("Smoke test completed.")
print(train_result)


/opt/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs


Starting smoke test...


<IPython.core.display.HTML object>

Smoke test completed.
TrainOutput(global_step=3, training_loss=3.5990240573883057, metrics={'train_runtime': 3.4748, 'train_samples_per_second': 3.453, 'train_steps_per_second': 0.863, 'total_flos': 2096565019776.0, 'train_loss': 3.5990240573883057, 'epoch': 0.0007406493025552401})


Reload the base model and create a fresh LoRA adapter

In [18]:
import gc
import torch
from pathlib import Path
from transformers import AutoModelForVision2Seq
from peft import LoraConfig, TaskType, get_peft_model

PROJECT_ROOT = Path("/home/jovyan/EdgeVLM_finetune")
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
MODEL_CACHE = PROJECT_ROOT / "model_cache"

# Release the model modified by the smoke test
if "model" in globals():
    del model

if "trainer" in globals():
    del trainer

gc.collect()
torch.cuda.empty_cache()

print("Reloading the original base model...")

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    cache_dir=str(MODEL_CACHE),
)

model = model.to("cuda")
model.config.use_cache = False

# Find attention layers in the text model
text_attention_targets = []

for name, module in model.named_modules():
    if (
        name.startswith("model.text_model.")
        and isinstance(module, torch.nn.Linear)
        and name.endswith(("q_proj", "k_proj", "v_proj", "o_proj"))
    ):
        text_attention_targets.append(name)

if not text_attention_targets:
    raise RuntimeError("No text-model attention layers found.")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=sorted(text_attention_targets),
)

model = get_peft_model(model, lora_config)
model.train()

print("Fresh LoRA model is ready.")
model.print_trainable_parameters()

Reloading the original base model...
Fresh LoRA model is ready.
trainable params: 921,600 || all params: 257,406,528 || trainable%: 0.3580


Start the real fine-tuning

In [19]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler
from transformers import TrainingArguments, Trainer

class BalancedTrainer(Trainer):
    def _get_train_sampler(self):
        labels = self.train_dataset.dataframe["label"].astype(str).tolist()
        counts = Counter(labels)

        weights = [
            1.0 / counts[label]
            for label in labels
        ]

        weights = torch.tensor(weights, dtype=torch.double)

        return WeightedRandomSampler(
            weights=weights,
            num_samples=len(weights),
            replacement=True,
        )

FINAL_OUTPUT = PROJECT_ROOT / "smolvlm_256m_lora"

training_args = TrainingArguments(
    output_dir=str(FINAL_OUTPUT),
    overwrite_output_dir=True,

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    fp16=torch.cuda.is_available(),

    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
    report_to="none",
)

trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
)

print("Starting real fine-tuning...")
train_result = trainer.train()

print("Fine-tuning completed.")
print(train_result)

trainer.save_model(str(FINAL_OUTPUT))
processor.save_pretrained(str(FINAL_OUTPUT))

print("Saved trained adapter to:", FINAL_OUTPUT)

/opt/conda/lib/python3.12/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Starting real fine-tuning...


<IPython.core.display.HTML object>

config.json: 0.00B [00:00, ?B/s]

Fine-tuning completed.
TrainOutput(global_step=2025, training_loss=0.19059213167355385, metrics={'train_runtime': 9109.3188, 'train_samples_per_second': 1.779, 'train_steps_per_second': 0.222, 'total_flos': 2833665790954752.0, 'train_loss': 0.19059213167355385, 'epoch': 0.9998765584495741})
Saved trained adapter to: /home/jovyan/EdgeVLM_finetune/smolvlm_256m_lora


Install evaluation tools

In [25]:
%pip install -q scikit-learn tqdm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Test the fine-tuned model

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

# Paths and data
FRAME_ROOT = Path("/home/jovyan")
PROJECT_ROOT = Path("/home/jovyan/EdgeVLM_finetune")

if "test_df" not in globals():
    test_df = pd.read_csv("/home/jovyan/splits/test.csv")

if "model" not in globals() or "processor" not in globals():
    raise RuntimeError(
        "The trained model and processor are not loaded. "
        "Run the model-loading and fine-tuning cells first."
    )

model.eval()

LABEL_ORDER = [
    "NORMAL",
    "DROWSY",
    "LOOKING_AWAY",
    "PHONE_USAGE",
]

UNSAFE_LABELS = {
    "DROWSY",
    "LOOKING_AWAY",
    "PHONE_USAGE",
}

# None means evaluate every test image
EVAL_LIMIT = None

if EVAL_LIMIT is None:
    eval_df = test_df.copy()
else:
    eval_df = test_df.head(EVAL_LIMIT).copy()

def parse_prediction(raw_text):
    """
    Returns:
    parsed_label: used for classification metrics
    exact_valid: true only when the raw output is exactly one allowed label
    """
    raw_text = str(raw_text).strip()
    normalized = raw_text.upper()

    exact_valid = normalized in LABEL_ORDER

    # Allow harmless punctuation for classification,
    # but do not count it as exact-label validity.
    cleaned = normalized.strip(" .,;:!?`'\"")

    if cleaned in LABEL_ORDER:
        return cleaned, exact_valid

    return "INVALID", False


def predict_one(row):
    image_path = FRAME_ROOT / str(row["frame_path"])

    with Image.open(image_path) as image_file:
        image = image_file.convert("RGB").copy()

    prompt = (
        "Classify the driver's visible state as exactly one label: "
        "NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE. "
        "Return only one label without explanation or punctuation."
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    chat_prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=chat_prompt,
        images=[image],
        return_tensors="pt",
        size={"longest_edge": 512},
        max_image_size={"longest_edge": 512},
    )

    device = next(model.parameters()).device
    inputs = inputs.to(device)

    start_time = time.perf_counter()

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
        )

    latency_ms = (time.perf_counter() - start_time) * 1000

    input_length = inputs["input_ids"].shape[-1]
    generated_only = generated[:, input_length:]

    raw_output = processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
    )[0].strip()

    parsed_label, exact_valid = parse_prediction(raw_output)

    return raw_output, parsed_label, exact_valid, latency_ms


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

true_labels = []
predicted_labels = []
raw_outputs = []
exact_valid_flags = []
latencies = []
paths = []

for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df),
    desc="Evaluating test images",
):
    raw_output, prediction, exact_valid, latency_ms = predict_one(row)

    true_labels.append(str(row["label"]).strip().upper())
    predicted_labels.append(prediction)
    raw_outputs.append(raw_output)
    exact_valid_flags.append(exact_valid)
    latencies.append(latency_ms)
    paths.append(str(row["frame_path"]))

true_labels = np.array(true_labels)
predicted_labels = np.array(predicted_labels)
exact_valid_flags = np.array(exact_valid_flags)
latencies = np.array(latencies)

# Per-class metrics for the four project classes
precision, recall, f1, support = precision_recall_fscore_support(
    true_labels,
    predicted_labels,
    labels=LABEL_ORDER,
    zero_division=0,
)

# Four-class balanced accuracy
balanced_accuracy_four_class = recall.mean()

# Error rates
unsafe_mask = np.isin(true_labels, list(UNSAFE_LABELS))
normal_mask = true_labels == "NORMAL"

unsafe_miss_rate = np.mean(
    predicted_labels[unsafe_mask] != true_labels[unsafe_mask]
)

false_alarm_rate = np.mean(
    np.isin(predicted_labels[normal_mask], list(UNSAFE_LABELS))
)

parser_failure_rate = np.mean(predicted_labels == "INVALID")
exact_label_validity_rate = exact_valid_flags.mean()

# Confusion matrix includes INVALID predictions
MATRIX_LABELS = LABEL_ORDER + ["INVALID"]

cm = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=MATRIX_LABELS,
)

# Save individual predictions for later analysis
results_df = eval_df.copy()
results_df["raw_output"] = raw_outputs
results_df["prediction"] = predicted_labels
results_df["exact_label_valid"] = exact_valid_flags
results_df["latency_ms"] = latencies

results_path = PROJECT_ROOT / "finetuned_test_predictions.csv"
results_df.to_csv(results_path, index=False)

# Report results
print("\n========== FINAL TEST RESULTS ==========")
print("Evaluation images:", len(eval_df))

print(
    "Exact-label validity rate:",
    round(exact_label_validity_rate * 100, 2),
    "%",
)

print(
    "Parser failure rate:",
    round(parser_failure_rate * 100, 2),
    "%",
)

print(
    "Overall accuracy:",
    round(
        accuracy_score(true_labels, predicted_labels) * 100,
        2,
    ),
    "%",
)

print(
    "Balanced accuracy:",
    round(balanced_accuracy_four_class * 100, 2),
    "%",
)

print(
    "Macro-F1:",
    round(
        f1_score(
            true_labels,
            predicted_labels,
            labels=LABEL_ORDER,
            average="macro",
            zero_division=0,
        ) * 100,
        2,
    ),
    "%",
)

print("\nPer-class results:")
for label, p, r, f, n in zip(
    LABEL_ORDER,
    precision,
    recall,
    f1,
    support,
):
    print(
        f"{label}: "
        f"precision={p:.3f}, "
        f"recall={r:.3f}, "
        f"F1={f:.3f}, "
        f"samples={n}"
    )

print("\nConfusion matrix:")
print(
    pd.DataFrame(
        cm,
        index=MATRIX_LABELS,
        columns=MATRIX_LABELS,
    )
)

print(
    "\nUnsafe-state miss rate:",
    round(unsafe_miss_rate * 100, 2),
    "%",
)

print(
    "False-alarm rate:",
    round(false_alarm_rate * 100, 2),
    "%",
)

print(
    "Mean latency (ms):",
    round(float(np.mean(latencies)), 2),
)

print(
    "P95 latency (ms):",
    round(float(np.percentile(latencies, 95)), 2),
)

if torch.cuda.is_available():
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print(
        "Peak GPU memory during evaluation (MB):",
        round(peak_memory_mb, 2),
    )

print("\nPredictions saved to:", results_path)

Evaluating test images:   0%|          | 0/3579 [00:00<?, ?it/s]